# Ready the ChatBot

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

model_id = "Qwen/Qwen2-0.5B-Instruct"

print(f"Loading model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto", # use float16 if GPU available
    device_map="auto"   # Automatically use GPU if present, else CPU
)


# 2. Create HF Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128, # Limit length for speed
    temperature=0.1, # Lower temperature for less creativity/hallucination 
    do_sample=True # Enable sampling if needed, but low temp makes it deterministic
)

# 3. LangChain LLM Wrapper

llm = HuggingFacePipeline(pipeline=pipe)

# 4. Define the Qwen Chat formatter
# This formats input as chat messages using Qwen's Templates.

def format_for_qwen(input_dict):
    messages = [
        { "role": "system", "content": """
You are a helpful AI assistant that answers questions strictly based on the provided context.
Do not use any external knowledge or make up information.
If the answer is not in the context, respond exactly with: "I don't know based on the provided context."
Always start your response with "Answer:" followed by the answer or the "I don't know" statement.
"""},
        {"role": "user", "content": f"""
Context:
{input_dict['context']}
Question: 
{input_dict["question"]}
"""}
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

# --- 5. Generation Function ---

def generate_with_qwen(formatted_prompt):
    # Use .invoke() instead of direct call
    response = llm.invoke(formatted_prompt)

    # Extract generated text (after prompt)
    generated = response.split(formatted_prompt)[-1].strip()

    # Further clean: If it starts with "Answer:", extract after it
    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[-1].strip()

    return generated


# --- 6. Create LLM Chain with Formatting ---

llm_with_format = (
    RunnableLambda(format_for_qwen)
    | RunnableLambda(generate_with_qwen)
    | StrOutputParser()
)

print("LLM setup complete!")

Loading model: Qwen/Qwen2-0.5B-Instruct...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 303.43it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


LLM setup complete!


# create a Flask APP

 ### Note: This app simply call the endpoint for chatbot

### Import The Libraries

In [15]:
%pip install flask flask-cors

Note: you may need to restart the kernel to use updated packages.


In [11]:
import threading
import time
import requests
from flask import Flask, request, jsonify
from flask_cors import CORS
import logging

### Environment Check

In [ ]:

if 'llm' not in globals() or 'tokenizer' not in globals():
    raise EnvironmentError("▲ ERROR: 'llmm' or 'tokenizer' not found. Please run the 'Model Setup' cell from the first lecture")

print("Environment check passed. LLM found.")

Environment check passed. LLM found.


### 1. Helper Functions

In [13]:
def create_qwen_prompt(user_text, history=None):
    if history is None:
        history = []

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."}
    ]

    messages.extend(history)
    messages.append({"role": "user", "content": user_text})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def clean_qwen_response(response: str, prompt: str) -> str:
    if prompt in response:
        return response.replace(prompt, "").strip()

    if "<|im_start|>assistant" in response:
        return response.split("<|im_start|>assistant")[-1].strip()

    return response.strip()

### 2. Flask App Setup

In [17]:
app = Flask(__name__)
CORS(app)    #Cross origin resource sharing
log = logging.getLogger('werkzeug') 
log.setLevel(logging.ERROR)

### 3. Routes

In [18]:
@app.route('/chat', methods=['POST'])
def chat():
    try:
        data = request.json
        formatted_prompt = create_qwen_prompt(data.get('message', ''))
        raw_response = llm.invoke(formatted_prompt)
        ai_response = clean_qwen_response(raw_response, formatted_prompt)
        return jsonify({"status": "success", "response": ai_response})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

### 4. Start Server

In [23]:
def run_flask():
    print("Flask Server started on http://127.0.0.1:5000")
    app.run(port=5000, use_reloader=False)

t = threading.Thread(target=run_flask)
t.daemon = True
t.start()

print("   \nWaiting for server to be ready...")
time.sleep(3)

Flask Server started on http://127.0.0.1:5000   
Waiting for server to be ready...

 * Serving Flask app '__main__'
 * Debug mode: off


### Test the General Chat

In [25]:
print("\n[TEST] Endpoint: /chatbot")
print("-" * 40)

try:
    resp = requests.post(
        'http://127.0.0.1:5000/chat',
        json={'message': 'Introduce yourself briefly.'} # User input here
    )

    if resp.status_code == 200:
        print(f"User: Introduce yourself briefly.")
        print(f"AI: {resp.json()['response']}")
    else:
        print(f"Error: {resp.status_code}")

except Exception as e:
    print(f"Failed: {e}")

time.sleep(1)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[TEST] Endpoint: /chatbot
----------------------------------------
User: Introduce yourself briefly.
AI: I am an artificial intelligence designed to assist with various tasks, including answering questions, providing information, and assisting in various activities. I am available 24/7 and can be used for various purposes such as language translation, text summarization, document analysis, and more. My purpose is to provide assistance and support to users who may need it.
